# 안면 감정 인식 앙상블 모델 추론 (Inference)

이 노트북은 새로운 이미지를 입력받아 **얼굴을 탐지(Face Detection)**하고, 사전에 학습된 여러 딥러닝 모델(VGG16, ResNet50 등)을 활용해 **감정(기쁨, 당황, 분노, 슬픔)을 추론**하며, 그 결과를 **시각화**하는 파이프라인을 제공합니다.

*   **구글 코랩(Google Colab)** 환경에 최적화되어 있습니다.
*   `MTCNN` 라이브러리를 사용하여 얼굴을 먼저 찾아낸 후, 모델에 알맞은 크기(224x224)로 잘라내어 전처리를 수행합니다.
*   여러 모델의 예측 확률을 평균 내는 **Soft Voting 앙상블 기법**을 사용하여 정확도를 높입니다.

### 1. 구글 드라이브 마운트 및 경로 설정
코랩 환경에서 구글 드라이브를 연결하고, 학습된 모델 파일(`.h5`)과 테스트할 이미지가 저장된 기본 경로를 설정합니다.

In [ ]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

import os

# 드라이브 내 프로젝트 기본 경로 설정 (실제 본인의 경로로 수정해주세요)
BASE_DIR = '/content/drive/MyDrive/Face.Sent'
MODEL_DIR = os.path.join(BASE_DIR, 'models') # 학습된 모델이 저장된 폴더
TEST_IMG_DIR = os.path.join(BASE_DIR, 'img/test') # 추론해볼 이미지가 있는 폴더

# 폴더가 없으면 생성 (경고 방지용)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TEST_IMG_DIR, exist_ok=True)
print(f"작업 경로 세팅 완료: {BASE_DIR}")

### 2. 필수 라이브러리 설치 및 임포트
얼굴을 정확하게 탐지하기 위한 `MTCNN` 라이브러리를 설치하고, 전처리와 모델 로드에 필요한 Keras/TensorFlow 라이브러리를 불러옵니다.

In [ ]:
# 2. 얼굴 탐지용 라이브러리(MTCNN) 설치 (코랩 환경)
!pip install mtcnn

# 필수 라이브러리 임포트
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from mtcnn import MTCNN
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import vgg16, resnet50, mobilenet

# 한글 폰트 깨짐 방지 (코랩용 임시 조치, 깨질 경우 아래 클래스 명을 영어로 변경하세요)
plt.rc('font', family='NanumBarunGothic') 

print("라이브러리 로드 완료!")

### 3. 전역 변수 설정 및 학습된 모델 불러오기
미리 학습시켜 구글 드라이브에 저장해둔 모델 파일(`VGG16_model.h5`, `model_ResNet.h5` 등)을 불러옵니다. 

⚠️ **주의**: 저장하신 파일 이름이 다르다면 아래 코드의 경로를 실제 파일명으로 수정해야 합니다.

In [ ]:
# 3. 감정 클래스 및 전역 변수 설정
# 시각화 시 한글이 깨지면 영문(['Happy', 'Panic', 'Angry', 'Sad'])으로 변경하세요.
EMOTIONS = ['기쁨', '당황', '분노', '슬픔']
IMG_SIZE = (224, 224) # 학습 시 사용했던 해상도

# 얼굴 탐지기 초기화
detector = MTCNN()

# 모델 불러오기 (사전에 학습하여 드라이브에 저장해둔 .h5 파일 경로)
# 주의: 파일명이 다를 경우 아래 경로를 실제 파일명으로 수정하세요.
try:
    print("모델 로딩 중... (시간이 조금 걸릴 수 있습니다.)")
    # VGG16 모델 로드
    model_vgg = load_model(os.path.join(MODEL_DIR, 'VGG16_model.h5'))
    # ResNet50 모델 로드
    model_resnet = load_model(os.path.join(MODEL_DIR, 'model_ResNet.h5'))
    
    # MobileNet 모델도 학습했다면 아래 주석을 해제하고 파일명을 맞춰주세요.
    # model_mobilenet = load_model(os.path.join(MODEL_DIR, 'MobileNet_model.h5')) 
    
    print("✅ 모든 모델을 성공적으로 불러왔습니다!")
except Exception as e:
    print(f"❌ 모델을 불러오는 중 오류 발생. 경로와 파일명을 확인하세요.\n에러내용: {e}")

### 4. 얼굴 탐지 및 맞춤형 이미지 전처리 파이프라인
임의의 이미지가 들어왔을 때 MTCNN을 이용해 얼굴 영역(Bounding Box)을 찾고, 여백을 주어 자른(Crop) 뒤, 각 사전 학습 모델에 맞는 전처리 함수(`preprocess_input`)를 통과시키는 함수입니다.

In [ ]:
# 4. 이미지 전처리 및 얼굴 크로핑 파이프라인
def detect_and_preprocess_face(image_path):
    # 이미지 로드 (OpenCV는 BGR이므로 RGB로 변환)
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("이미지를 찾을 수 없거나 열 수 없습니다.")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # 얼굴 탐지 수행
    faces = detector.detect_faces(img_rgb)
    
    if not faces:
        print("⚠️ 얼굴을 감지하지 못했습니다. 원본 이미지를 중앙 기준으로 크롭하여 사용합니다.")
        h_img, w_img = img_rgb.shape[:2]
        # 임의로 중앙 영역 크롭
        min_dim = min(h_img, w_img)
        y1 = (h_img - min_dim) // 2
        x1 = (w_img - min_dim) // 2
        cropped_face = img_rgb[y1:y1+min_dim, x1:x1+min_dim]
        box = (x1, y1, min_dim, min_dim)
    else:
        # 가장 크게 감지된 첫 번째 얼굴 선택
        x, y, w, h = faces[0]['box']
        # 여백을 약간 주고 자르기 (너무 타이트하게 잘리는 것 방지)
        pad = int(max(w, h) * 0.1)
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(img_rgb.shape[1], x + w + pad)
        y2 = min(img_rgb.shape[0], y + h + pad)
        
        cropped_face = img_rgb[y1:y2, x1:x2]
        box = (x1, y1, x2-x1, y2-y1) # 시각화를 위한 박스 정보
        
    # 모델 입력 크기에 맞게 리사이즈
    resized_face = cv2.resize(cropped_face, IMG_SIZE)
    img_array = np.expand_dims(resized_face, axis=0) # 배치 차원 추가 (1, 224, 224, 3)
    img_array = np.array(img_array, dtype=np.float32)

    # 각 모델별 전처리 수행
    # Keras의 사전 학습 모델들은 각자 고유의 전처리 방식(Scaling, Mean Subtraction 등)을 가짐
    input_vgg = vgg16.preprocess_input(img_array.copy())
    input_resnet = resnet50.preprocess_input(img_array.copy())
    
    # MobileNet 전처리가 필요하다면 아래 주석 해제
    # input_mobilenet = mobilenet.preprocess_input(img_array.copy())
    # return img_rgb, box, input_vgg, input_resnet, input_mobilenet
    
    return img_rgb, box, input_vgg, input_resnet

### 5. 앙상블 추론 및 시각화 함수
전처리된 이미지를 각 모델에 통과시켜 확률을 예측하고, 이를 평균 내어(Soft Voting) 최종 감정을 판단합니다. 원본 이미지 위에 Bounding Box를 그리고 옆에 확률 분포 막대 그래프를 표시합니다.

In [ ]:
# 5. 앙상블 예측 및 결과 시각화
def predict_and_visualize(image_path):
    try:
        # 전처리 파이프라인 실행
        # 모바일넷을 추가했다면 반환값을 하나 더 받도록 수정하세요.
        original_img, box, input_vgg, input_resnet = detect_and_preprocess_face(image_path)
        
        # 1. 개별 모델 예측 (확률값 반환)
        pred_vgg = model_vgg.predict(input_vgg, verbose=0)[0]
        pred_resnet = model_resnet.predict(input_resnet, verbose=0)[0]
        
        # 모바일넷 예측 코드가 필요하면 주석 해제
        # pred_mobilenet = model_mobilenet.predict(input_mobilenet, verbose=0)[0]
        
        # 2. 앙상블 (Soft Voting: 예측 확률의 평균 계산)
        # 모바일넷 포함 시 (pred_vgg + pred_resnet + pred_mobilenet) / 3.0 으로 변경
        ensemble_pred = (pred_vgg + pred_resnet) / 2.0 
        
        # 최종 감정 도출
        final_class_idx = np.argmax(ensemble_pred)
        final_emotion = EMOTIONS[final_class_idx]
        confidence = ensemble_pred[final_class_idx] * 100
        
        # 3. 시각화
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # [왼쪽] 원본 이미지에 박스와 텍스트 표시
        img_draw = original_img.copy()
        x, y, w, h = box
        cv2.rectangle(img_draw, (x, y), (x+w, y+h), (0, 255, 0), 3) # 초록색 Bounding Box
        
        ax1.imshow(img_draw)
        ax1.axis('off')
        ax1.set_title(f"예측 결과: {final_emotion} ({confidence:.1f}%)", fontsize=16, fontweight='bold', color='red')
        
        # [오른쪽] 예측 확률 막대 그래프
        y_pos = np.arange(len(EMOTIONS))
        colors = ['green' if i == final_class_idx else 'gray' for i in range(len(EMOTIONS))]
        
        ax2.barh(y_pos, ensemble_pred, color=colors)
        ax2.set_yticks(y_pos)
        ax2.set_yticklabels(EMOTIONS, fontsize=12)
        ax2.invert_yaxis()  # 위에서부터 아래로 읽기 편하게 반전
        ax2.set_xlabel('확률 (Probability)')
        ax2.set_title('앙상블 모델 확률 분포', fontsize=14)
        ax2.set_xlim(0, 1.0)
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"추론 실패: {e}")

### 6. 실제 이미지로 테스트 실행
테스트할 이미지의 경로를 지정하고 추론 함수를 실행합니다. 코랩 좌측의 '파일' 메뉴를 통해 이미지를 업로드한 후, 해당 경로를 입력하세요.

In [ ]:
# 6. 실제 이미지로 테스트 실행
# 테스트할 이미지를 코랩에 업로드하거나 드라이브의 경로를 입력하세요.
test_img_path = '/content/test_image.jpg' # 예시 경로

# 드라이브에 있는 테스트 이미지를 사용하려면 아래처럼 작성하세요.
# test_img_path = os.path.join(TEST_IMG_DIR, '테스트할_이미지_이름.jpg')

# 이미지 파일이 실제로 존재하는지 확인 후 실행
if os.path.exists(test_img_path):
    predict_and_visualize(test_img_path)
else:
    print(f"이미지 파일을 찾을 수 없습니다: {test_img_path}\n좌측 폴더 모양 아이콘을 클릭하여 이미지를 업로드해주세요.")